# Pelatihan & Kuantisasi CNN Deteksi Masker Wajah (QAT Power-of-Two ±2^k)
**Nama:** Muhammad Gazali  
**NIM:** 101022330164  
**Mata Kuliah:** Desain SoC dan FPGA  

Untuk dataset **Face Mask 12K Images** (`ashishjangra27/face-mask-12k-images-dataset`), pendekatan paling optimal untuk akselerator FPGA berdaya rendah adalah menggunakan input **Grayscale 1-channel (`1 × 32 × 32`)**. Selain mempermudah ekstraksi fitur kontras area masker dan wajah terbuka, format 1-channel memangkas memori buffer input hingga 3× lipat dibandingkan citra RGB saat disintesis ke High-Level Synthesis (HLS).

Dataset berisi dua kategori seimbang: **WithMask** (`5.883` citra) dan **WithoutMask** (`5.909` citra) dengan total **11.792 citra** yang dibagi menjadi **80% Training** (`9.433` citra) dan **20% Validation/Test** (`2.359` citra).

---
## Langkah 1: Unduh & Siapkan Dataset Face Mask 12K di Colab
Jalankan sel ini terlebih dahulu. Menggunakan `kagglehub`, dataset langsung terunduh ke Colab tanpa perlu mengunggah file `kaggle.json`:

In [1]:
import os
import shutil
import kagglehub

# 1. Download dataset Face Mask 12K langsung dari Kaggle
path = kagglehub.dataset_download("ashishjangra27/face-mask-12k-images-dataset")
print("Dataset terunduh di:", path)

# 2. Susun ke folder data_raw/WithMask dan data_raw/WithoutMask
base_folder = os.path.join(path, "Face Mask Dataset")
os.makedirs("data_raw/WithMask", exist_ok=True)
os.makedirs("data_raw/WithoutMask", exist_ok=True)

for split in ["Train", "Validation", "Test"]:
    for cls in ["WithMask", "WithoutMask"]:
        src_dir = os.path.join(base_folder, split, cls)
        dst_dir = os.path.join("data_raw", cls)
        if os.path.exists(src_dir):
            for fname in os.listdir(src_dir):
                shutil.copy(os.path.join(src_dir, fname), os.path.join(dst_dir, f"{split}_{fname}"))

print("Jumlah WithMask:", len(os.listdir("data_raw/WithMask")))
print("Jumlah WithoutMask:", len(os.listdir("data_raw/WithoutMask")))

---
## Langkah 2: Arsitektur `TinyMaskCNN_32x32` & Quantization-Aware Training (QAT PoT)
Pada tahap ini, seluruh bobot konvolusi dan *fully connected* dikuantisasi selama proses *training* menggunakan **Straight-Through Estimator (STE)** menjadi bilangan pangkat dua ($\{0, \pm 2^{-7}, \dots, \pm 2^0\}$). Pendekatan ini memungkinkan operasi perkalian pada perangkat keras digantikan sepenuhnya oleh operasi *bit-shift* (`>>`) tanpa menggunakan blok DSP FPGA.

In [2]:
import glob
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

# Loader Dataset (32x32 Grayscale)
class MaskDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        for folder, label in [("WithMask", 0), ("WithoutMask", 1)]:
            for p in glob.glob(os.path.join(root_dir, folder, "*.*")):
                self.samples.append((p, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("L").resize((32, 32))
        arr = np.array(img, dtype=np.float32) / 255.0
        return torch.tensor(arr).unsqueeze(0), label

# Kuantisasi Power-of-Two (±2^k) dengan Straight-Through Estimator
class PoTQuantizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w):
        sign = torch.sign(w)
        abs_w = torch.abs(w)
        mask = (abs_w >= (2.0 ** -7.5)).float()
        exp = torch.clamp(torch.round(torch.log2(abs_w + 1e-12)), -7, 0)
        return sign * (2.0 ** exp) * mask

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

class PoTConv2d(nn.Conv2d):
    def forward(self, x):
        return nn.functional.conv2d(x, PoTQuantizeSTE.apply(self.weight), self.bias, self.stride, self.padding)

class PoTLinear(nn.Linear):
    def forward(self, x):
        return nn.functional.linear(x, PoTQuantizeSTE.apply(self.weight), self.bias)

class MaskCNN_PoT(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = PoTConv2d(1, 4, kernel_size=5, stride=1, padding=0, bias=False)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool2d(kernel_size=4, stride=4)
        self.fc    = PoTLinear(4 * 7 * 7, 2, bias=False)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

# Pembagian Data 80:20 & Proses Training QAT
dataset = MaskDataset("data_raw")
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)

model = MaskCNN_PoT()
optimizer = optim.Adam(model.parameters(), lr=0.004)
criterion = nn.CrossEntropyLoss()

for epoch in range(15):
    model.train()
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

model.eval()
correct = 0
with torch.no_grad():
    for x, y in val_loader:
        correct += (model(x).argmax(1) == y).sum().item()
print(f"Akurasi Validasi QAT Power-of-Two: {100.0 * correct / val_len:.2f}%")
torch.save(model.state_dict(), "model_masker_32x32_pot.pth")

---
## Langkah 3: Ekspor Header C++ (`weights.h` & `test_samples.h`) untuk HLS
Sel ini mengonversi bobot Power-of-Two terlatih menjadi kode pergeseran bit (`int8_t`) ke dalam `weights.h` dan mengekspor 10 citra validasi ke format *Fixed-Point Q4.12* (`int32_t`) pada `test_samples.h` untuk diverifikasi pada modul `hls/cnn_hls.cpp`.

In [3]:
def encode_pot_shift(tensor):
    q_w = PoTQuantizeSTE.apply(tensor).detach().numpy()
    encoded = np.zeros_like(q_w, dtype=np.int8)
    for idx, val in np.ndenumerate(q_w):
        if val != 0.0:
            s = 1 if val > 0 else -1
            exp = int(round(np.log2(abs(val))))
            encoded[idx] = s * (exp + 8)
    return encoded

w_conv1_enc = encode_pot_shift(model.conv1.weight.data)
w_fc_enc    = encode_pot_shift(model.fc.weight.data)
print("Total Bobot Terkuantisasi PoT:", w_conv1_enc.size + w_fc_enc.size, "Bytes")